In [2]:
import duckdb
import pandas as pd
import os

In [3]:
# 将 CSV 数据导入数据库
csv_path = os.path.abspath('data/通达信数据_20251229.csv')
# DuckDB文件路径
db_path = os.path.abspath('data/通达信数据_20251229.duckdb')
conn=duckdb.connect(db_path)
conn.execute(f"""
    CREATE OR REPLACE TABLE stock_data AS 
    SELECT * FROM read_csv('{csv_path}')
""")
print("数据导入成功！")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

数据导入成功！


In [4]:
# 获取表名，验证导入是否成功
tables = conn.execute('SHOW TABLES').fetchall()
print("数据库中的表：", tables)

# 查看表结构
if tables:
    print("\n表 'stock_data' 的前5行数据：")
    result = conn.execute('SELECT * FROM stock_data LIMIT 5').fetchdf()
    print(result)
    print(f"\n总行数: {conn.execute('SELECT COUNT(*) as count FROM stock_data').fetchone()[0]}")


数据库中的表： [('stock_data',)]

表 'stock_data' 的前5行数据：
   代码    名称         日期   涨幅%     现价     今开    涨跌  一年涨幅%  利润同比%  10日涨幅%  ...  \
0   1  平安银行 2025-02-12  0.00  11.42  11.41  0.00  31.11   0.24    0.79  ...   
1   1  平安银行 2025-02-13  0.70  11.50  11.42  0.08  32.03   0.24    3.70  ...   
2   1  平安银行 2025-02-14  0.43  11.55  11.49  0.05  32.60   0.24    2.03  ...   
3   1  平安银行 2025-02-17  1.99  11.78  11.60  0.23  35.25   0.24    3.88  ...   
4   1  平安银行 2025-02-18  0.25  11.81  11.76  0.03  35.59   0.24    2.96  ...   

   相对行业强度     分类价格指数    分类日收益率   分类5日收益率  分类20日收益率     分类波动率      分类RSI  分类动量  \
0   -1.43  21.140993  0.000000  0.000000  0.000000  0.000000   0.000000   0.0   
1    1.47  21.167873  0.001271  0.001271  0.001271  0.000000   0.000000   0.0   
2   -0.72  21.394063  0.010686  0.005979  0.005979  0.006657  99.999997   0.0   
3    1.60  21.182256 -0.009900  0.000686  0.000686  0.010305  31.380377   0.0   
4    1.87  21.322595  0.006625  0.002171  0.002171  0.008923  55.75370

In [5]:
#获取列名
table_name=tables[0][0]
columns=conn.execute(f"DESCRIBE {table_name}").fetchall()
all_columns=[col[0] for col in columns]
print(f"所有列名：{all_columns}")

所有列名：['代码', '名称', '日期', '涨幅%', '现价', '今开', '涨跌', '一年涨幅%', '利润同比%', '10日涨幅%', '换手%', '总金额', '流通市值', '股息率%', '量比', '年初至今%', '总量', '现量', '最高', '最低', '昨收', '市盈(动)', '振幅%', '均价', '内盘', '外盘', '内外比', '买量', '卖量', '委比%', '量涨速%', '主力净额', '主力净比%', '昨成交额', '开盘金额', '开盘抢筹%', '开盘昨比%', '开盘换手Z', '竞价量比', '封成比', '封单额', '流通股(亿)', 'AB股总市值', '强弱度%', '活跃度', '连涨天', '昨涨幅%', '3日涨幅%', '5日涨幅%', '20日涨幅%', '60日涨幅%', '月初至今%', '年涨停天', '流通股本Z', '换手Z', '流通市值Z', '市值增减', '市盈(TTM)', '市盈(静)', '贝塔系数', '距5日线%', '近日指标提示', '短期形态', '中期形态', '长期形态', '开盘%', '最高%', '最低%', '均涨幅%', '回头波%', '攻击波%', '总股本(亿)', 'B/A股(亿)', 'H股(亿)', '总资产(亿)', '净资产(亿)', '少数股权(亿)', '资产负债率%', '流动资产(亿)', '固定资产(亿)', '无形资产(亿)', '流动负债(亿)', '货币资金(亿)', '存货(亿)', '应收账款(亿)', '合同负债(亿)', '资本公积金(亿)', '营业收入(亿)', '营业成本(亿)', '营业利润(亿)', '投资收益(亿)', '利润总额(亿)', '净利润(亿)', '扣非净利润(亿)', '未分利润(亿)', '经营现金流(亿)', '总现金流(亿)', '股东人数', '人均持股', '人均市值', '收入同比%', '市净率', '市现率', '市销率', '每股收益', '每股净资', '每股公积', '每股未分配', '每股现金流', '权益比%', '净益率%', '毛利率%', '营业利润率%', '净利润率%', '研发费用(亿)', '员工人数', '主力占比%

In [6]:
#查找与价格有关的列
price_columns=[col for col in all_columns if any(keyword in col for keyword in['价格', '价', '收盘', '开盘', '最高', '最低'])]
print(f"与价格有关的列{price_columns}")

与价格有关的列['现价', '最高', '最低', '均价', '开盘金额', '开盘抢筹%', '开盘昨比%', '开盘换手Z', '竞价量比', '开盘%', '最高%', '最低%', '昨开盘金额', '发行价', '52周最高', '52周最低', '收盘价', '竞价昨比', '开盘量比', '行业价格指数', '分类价格指数']


In [7]:
volume_columns=[col for col in all_columns if any(keyword in col for keyword in['量', '成交量', '总量'])]
print(f"与价格有关的列{volume_columns}")

与价格有关的列['量比', '总量', '现量', '买量', '卖量', '量涨速%', '竞价量比', '笔均量', '总委量差', '逐笔均量', '开盘量比', '行业动量', '分类动量']


In [8]:
# 关闭数据库连接
conn.close()